# SCBF Training on Google Colab

This notebook trains the SCBF (Supply Chain Behavioral Fingerprinting) model on your dataset.

**Runtime:** Use GPU for faster training (Runtime > Change runtime type > GPU)

**Dataset:** Upload your `data/zenodo_13746167/` folder or mount Google Drive

## Step 1: Setup Environment

In [ ]:
# Install dependencies
!pip install torch torchvision numpy jsonlines scikit-learn matplotlib -q

print("✓ Dependencies installed")

## Step 2: Upload Dataset

**Option A: Upload from local machine**
- Zip your `data/zenodo_13746167/` folder first
- Upload the zip using the file browser (left sidebar)
- Uncomment and run the cell below

**Option B: Mount Google Drive**
- Upload data to Google Drive first
- Mount drive and copy to Colab

In [ ]:
# Option A: Extract uploaded zip
# !unzip -q data.zip
# !ls -la data/zenodo_13746167/benign/traces/ | head -5
# !ls -la data/zenodo_13746167/malware/traces/ | head -5

# Option B: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy from Drive (adjust path as needed)
# !cp -r /content/drive/MyDrive/SCBF/data .

# Verify data
!echo "Benign traces:"
!ls data/zenodo_13746167/benign/traces/*.jsonl 2>/dev/null | wc -l
!echo "Malware traces:"
!ls data/zenodo_13746167/malware/traces/*.jsonl 2>/dev/null | wc -l

## Step 3: Clone Repository (for model code)

In [ ]:
# Clone your SCBF repository
!git clone https://github.com/ritik-roushan-rana/SCBF.git
%cd SCBF

# Copy data to repo
!mkdir -p data/zenodo_13746167
!cp -r ../data/zenodo_13746167/* data/zenodo_13746167/

print("✓ Repository cloned and data copied")

## Step 4: Verify Data

In [ ]:
import glob
import json

benign_files = glob.glob("data/zenodo_13746167/benign/traces/*.jsonl")
malware_files = glob.glob("data/zenodo_13746167/malware/traces/*.jsonl")

print(f"Found {len(benign_files)} benign packages")
print(f"Found {len(malware_files)} malware packages")
print(f"Total: {len(benign_files) + len(malware_files)} packages")

# Test load one file
if len(benign_files) > 0:
    with open(benign_files[0], 'r') as f:
        events = [json.loads(line) for line in f]
    print(f"\nSample file has {len(events)} events")
    print(f"First event: {events[0]}")

assert len(benign_files) > 0, "No benign files found!"
assert len(malware_files) > 0, "No malware files found!"
print("\n✓ Data verified!")

## Step 5: Check GPU Availability

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("\n✓ GPU training will be FAST!")
else:
    print("\n⚠ Using CPU - training will be slow")
    print("Change runtime: Runtime > Change runtime type > GPU")

## Step 6: Train Model

This will train for up to 60 epochs with early stopping.

**On GPU:** ~30-60 minutes  
**On CPU:** ~2-3 hours

In [ ]:
# Import training script
import sys
sys.path.insert(0, '/content/SCBF')

# Run training
!python -m scbf.training.train_with_split

## Step 7: Quick Evaluation

In [ ]:
import torch
import json
import numpy as np
from scbf.models.tgn_encoder import TGNEncoder
from scbf.models.itbg_constructor import ITBGConstructor

# Load split info
with open("models/checkpoints/split_info.json", 'r') as f:
    split_info = json.load(f)

test_data = split_info['test']

print(f"Test set: {len(test_data)} samples")
print(f"  Clean: {sum(1 for x in test_data if x['label'] == 0)}")
print(f"  Malicious: {sum(1 for x in test_data if x['label'] == 1)}")

# Load model
print("\nLoading model...")
model = TGNEncoder(num_nodes=50000)
model.load_state_dict(torch.load("models/tgn_v2_best.pt"))
model.eval()

# Move to GPU if available
if torch.cuda.is_available():
    model = model.cuda()

def load_events(path):
    with open(path, 'r') as f:
        return [json.loads(line) for line in f]

# Compute embeddings
print("\nComputing embeddings...")
embeddings_list = []
labels_list = []

for i, item in enumerate(test_data):
    if (i + 1) % 50 == 0:
        print(f"  Processed {i + 1}/{len(test_data)}...")
    
    path = item['path']
    label = item['label']
    
    model.memory_bank.reset_memory()
    constructor = ITBGConstructor(model)
    
    try:
        events = load_events(path)
        dna = constructor.replay_session(events)
        
        if dna is not None:
            embeddings_list.append(dna.detach().cpu().numpy())
            labels_list.append(label)
    except Exception as e:
        continue

embeddings = np.array(embeddings_list)
labels = np.array(labels_list)

# Compute distances
clean_mask = labels == 0
clean_centroid = embeddings[clean_mask].mean(axis=0)
distances = np.sqrt(((embeddings - clean_centroid) ** 2).sum(axis=1))

clean_dists = distances[labels == 0]
mal_dists = distances[labels == 1]

print("\n" + "=" * 80)
print("DISTANCE STATISTICS")
print("=" * 80)
print(f"Clean distances: mean={clean_dists.mean():.4f}, std={clean_dists.std():.4f}")
print(f"Malicious distances: mean={mal_dists.mean():.4f}, std={mal_dists.std():.4f}")
print(f"Separation: {mal_dists.mean() - clean_dists.mean():.4f}")

# Auto-threshold
threshold = clean_dists.mean() + 2 * clean_dists.std()
print(f"\nAuto-computed threshold: {threshold:.4f}")

# Predictions
predictions = (distances > threshold).astype(int)

# Metrics
tp = ((predictions == 1) & (labels == 1)).sum()
tn = ((predictions == 0) & (labels == 0)).sum()
fp = ((predictions == 1) & (labels == 0)).sum()
fn = ((predictions == 0) & (labels == 1)).sum()

accuracy = (tp + tn) / len(labels)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("\n" + "=" * 80)
print("TEST SET RESULTS")
print("=" * 80)
print(f"\nAccuracy:  {accuracy:.2%} ({accuracy:.4f})")
print(f"Precision: {precision:.2%} ({precision:.4f})")
print(f"Recall:    {recall:.2%} ({recall:.4f})")
print(f"F1 Score:  {f1:.2%} ({f1:.4f})")

print(f"\nConfusion Matrix:")
print(f"                Predicted")
print(f"              Clean  Malicious")
print(f"Actual Clean    {tn:3d}  {fp:3d}")
print(f"Actual Mal      {fn:3d}  {tp:3d}")

# AUC
try:
    from sklearn.metrics import roc_auc_score
    auc = roc_auc_score(labels, distances)
    print(f"\nROC-AUC: {auc:.4f}")
except:
    pass

print("\n" + "=" * 80)

## Step 8: Download Trained Model

In [ ]:
from google.colab import files

# Download best model
files.download('models/tgn_v2_best.pt')

# Download checkpoints (optional)
# !zip -r checkpoints.zip models/checkpoints/
# files.download('checkpoints.zip')

print("✓ Model downloaded!")

## Summary

**What you trained:**
- Model: TGN (Temporal Graph Network) with improved contrastive loss
- Dataset: 1,344 packages (959 benign, 385 malware)
- Split: 70% train, 15% val, 15% test
- Epochs: Up to 60 with early stopping (patience=10)

**Key metrics to check:**
- Malicious distance: Should be > 1.5 (good separation)
- Accuracy: Target > 85%
- Recall: Target > 70%
- F1 Score: Target > 75%

**Next steps:**
1. Download the trained model (`tgn_v2_best.pt`)
2. Copy it to your local `models/` directory
3. Build envelope: `python -m scbf.training.build_envelope`
4. Scan packages: `make scan PKG=requests`